# Synapse Engine: Qwen 3B Fine-Tuning & GGUF Export
This notebook will:
1. Train the model using QLoRA on your dataset.
2. Merge the LoRA weights.
3. Convert the model to Ollama-compatible `.gguf` format.
4. Upload the result directly to your Hugging Face account.

In [ ]:
!pip install -q -U torch transformers peft trl datasets bitsandbytes accelerate huggingface_hub

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
except Exception as e:
    print("⚠️ Please add HF_TOKEN to your Kaggle Secrets to upload the final model!")

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# You can adjust this to any Qwen 3B-4B class model
model_id = "Qwen/Qwen2.5-3B-Instruct"
output_dir = "/kaggle/working/synapse-qwen-finetuned"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto"
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16, lora_alpha=32, 
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

In [ ]:
# Update these paths if you named your Kaggle Dataset differently
dataset = load_dataset("json", data_files={
    "train": "/kaggle/input/synapse-dataset/train.jsonl",
    "test": "/kaggle/input/synapse-dataset/val.jsonl"
})

def format_chat(example):
    example["text"] = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return example

dataset = dataset.map(format_chat)

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
    max_grad_norm=0.3,
    max_steps=300,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
)

print("Starting Training...")
trainer.train()
trainer.model.save_pretrained(f"{output_dir}/adapter")
tokenizer.save_pretrained(f"{output_dir}/adapter")
print("Training Complete!")

In [ ]:
print("Merging model and preparing for GGUF conversion...")
import gc
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

from peft import AutoPeftModelForCausalLM
merged_model = AutoPeftModelForCausalLM.from_pretrained(
    f"{output_dir}/adapter", torch_dtype=torch.float16, device_map="cpu"
)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained(f"{output_dir}/merged")
tokenizer.save_pretrained(f"{output_dir}/merged")

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
!pip install -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py {output_dir}/merged --outfile /kaggle/working/synapse-qwen-4b.gguf --outtype q8_0

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

# REPLACE 'your-username' WITH YOUR HUGGINGFACE USERNAME
hf_username = "your-username" 
repo_id = f"{hf_username}/synapse-qwen-4b-gguf"

try:
    api.create_repo(repo_id=repo_id, exist_ok=True)
    print("Uploading GGUF to HuggingFace...")
    api.upload_file(
        path_or_fileobj="/kaggle/working/synapse-qwen-4b.gguf",
        path_in_repo="synapse-qwen-4b.gguf",
        repo_id=repo_id
    )
    print(f"Success! You can now download the model from https://huggingface.co/{repo_id}")
except Exception as e:
    print(f"Upload failed (did you set HF_TOKEN and your username?): {e}")